# Bid Offer Stack Source

In [50]:
from pathlib import Path
import io
import csv
import zipfile
import traceback
import requests
import nemosis
import pandas as pd
import numpy as np


In [51]:
# # Load per-region aggregated energy bid offer stack and late-rebid activity features from BIDPEROFFER_D + BIDDAYOFFER_D.

# # Features produced per SETTLEMENTDATE (5-min) per region:
# # - `bidstack_mw_neg`   — MW offered at negative prices (must-run / overflow)
# # - `bidstack_mw_low`   — MW offered $0–$300 (baseload)
# # - `bidstack_mw_mid`   — MW offered $300–$1 000
# # - `bidstack_mw_high`  — MW offered $1 000–$5 000
# # - `bidstack_mw_vhigh` — MW offered $5 000–$14 500
# # - `bidstack_mw_cap`   — MW offered ≥$14 500 (VoLL / price cap)
# # - `bidstack_mw_total` — total MAXAVAIL across all DUIDs
# # - `bidstack_rebid_count` — number of DUID-level late rebids (LASTCHANGED within 30 min of dispatch interval)



# BAND_COLS  = [f"BANDAVAIL{i}" for i in range(1, 11)]
# PRICE_COLS = [f"PRICEBAND{i}" for i in range(1, 11)]

# PRICE_TIERS = [
#     ("neg",    -np.inf,    0.0),
#     ("low",       0.0,   300.0),
#     ("mid",     300.0,  1000.0),
#     ("high",   1000.0,  5000.0),
#     ("vhigh",  5000.0, 14500.0),
#     ("cap",   14500.0,  np.inf),
# ]
# TIER_NAMES = [t for t, _, _ in PRICE_TIERS]

# RAW_DIR = Path("Pre_processing/temporary_cache")

# # ── AEMO filename format history ───────────────────────────────────────────────
# # BIDPEROFFER:
# #   2018-01 → 2021-02  PUBLIC_DVD_BIDPEROFFER_D_*     ~100-200 MB  ← nemosis OK
# #   2021-03 → 2022-05  PUBLIC_DVD_BIDPEROFFER_*       ~636-1562 MB, cumulative
# #   2022-06 → 2024-07  PUBLIC_DVD_BIDPEROFFER1_* +    cumulative, split into 2
# #                      PUBLIC_DVD_BIDPEROFFER2_*         ~1-7 GB total
# #   2024-08 →          PUBLIC_ARCHIVE#BIDPEROFFER_D#FILE01#*        ← nemosis OK
# # BIDDAYOFFER:
# #   2018-01 → 2021-02  PUBLIC_DVD_BIDDAYOFFER_D_*     ~100 MB      ← nemosis OK
# #   2021-03 → 2024-07  PUBLIC_DVD_BIDDAYOFFER_*       ~128 MB      ← direct download
# #   2024-08 →          PUBLIC_ARCHIVE#BIDDAYOFFER_D#FILE01#*        ← nemosis OK
# #
# # Gap strategy: stream to disk, parse row-by-row keeping only ENERGY rows
# # for the target month (TRADINGDATE prefix filter — files are cumulative).
# # Deduplication (latest OFFERDATETIME per DUID/TRADINGDATE/PERIODID) happens
# # DURING parsing via a dict so we never accumulate all rebid versions in memory.
# # ──────────────────────────────────────────────────────────────────────────────
# GAP_START       = pd.Timestamp("2021-03-01")
# GAP_SPLIT_START = pd.Timestamp("2022-06-01")   # when BIDPEROFFER split into 1+2
# GAP_END         = pd.Timestamp("2024-08-01")

# _AEMO_MMS_BASE = (
#     "https://nemweb.com.au/Data_Archive/Wholesale_Electricity/MMSDM"
#     "/{year}/MMSDM_{year}_{month:02d}/MMSDM_Historical_Data_SQLLoader/DATA"
# )


# def _trading_day(times: pd.Series) -> pd.Series:
#     d = times.dt.normalize()
#     return d.where(times.dt.hour >= 4, d - pd.Timedelta(days=1))


# def _fetch_duid_region() -> dict:
#     df = nemosis.dynamic_data_compiler(
#         start_time="2018/01/01 00:00:00",
#         end_time="2026/01/01 00:00:00",
#         table_name="DUDETAILSUMMARY",
#         raw_data_location=str(RAW_DIR),
#         select_columns=["DUID", "REGIONID", "START_DATE", "END_DATE"],
#         fformat="feather",
#         keep_csv=False,
#         parse_data_types=False,
#     )
#     df.columns = [c.upper() for c in df.columns]
#     df["START_DATE"] = pd.to_datetime(df["START_DATE"], errors="coerce")
#     return (
#         df.dropna(subset=["START_DATE"])
#         .sort_values("START_DATE")
#         .groupby("DUID")["REGIONID"]
#         .last()
#         .str.lower()
#         .to_dict()
#     )


# def _cleanup_month_raw(month: pd.Timestamp) -> None:
#     pattern = month.strftime("%Y%m")
#     for f in RAW_DIR.glob(f"*{pattern}*"):
#         if f.is_file():
#             f.unlink()


# def _stream_download(url: str, dest: Path) -> bool:
#     try:
#         r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, stream=True, timeout=600)
#         if r.status_code != 200:
#             return False
#         with open(dest, "wb") as fh:
#             for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
#                 fh.write(chunk)
#         return True
#     except Exception:
#         return False


# def _parse_mms_energy_zip(
#     path: Path,
#     keep_cols: set,
#     filter_ym: str | None = None,
#     dedup_key: tuple | None = None,
#     dedup_max: str | None = None,
# ) -> pd.DataFrame:
#     header = None
#     keep_idx = []
#     bidtype_i = tradingdate_i = dedup_max_i = None
#     dedup_key_idx = []
#     use_dedup = dedup_key is not None and dedup_max is not None
#     best = {}
#     rows = []
#     with zipfile.ZipFile(path) as z:
#         with z.open(z.namelist()[0]) as f:
#             for parts in csv.reader(io.TextIOWrapper(f, encoding="utf-8", errors="replace")):
#                 if not parts:
#                     continue
#                 tag = parts[0]
#                 if tag == "I":
#                     full_hdr = parts[4:]
#                     keep_idx = [i for i, c in enumerate(full_hdr) if c in keep_cols]
#                     header = [full_hdr[i] for i in keep_idx]
#                     bidtype_i     = full_hdr.index("BIDTYPE")     if "BIDTYPE"     in full_hdr else None
#                     tradingdate_i = full_hdr.index("TRADINGDATE") if "TRADINGDATE" in full_hdr else None
#                     if use_dedup:
#                         dedup_key_idx = [full_hdr.index(c) for c in dedup_key if c in full_hdr]
#                         dedup_max_i   = full_hdr.index(dedup_max) if dedup_max in full_hdr else None
#                 elif tag == "D" and header is not None:
#                     data = parts[4:]
#                     if bidtype_i is not None and bidtype_i < len(data) and data[bidtype_i] != "ENERGY":
#                         continue
#                     if (filter_ym and tradingdate_i is not None
#                             and tradingdate_i < len(data)
#                             and not data[tradingdate_i].startswith(filter_ym)):
#                         continue
#                     slim_row = [data[i] if i < len(data) else "" for i in keep_idx]
#                     if use_dedup:
#                         key = tuple(data[i] if i < len(data) else "" for i in dedup_key_idx)
#                         val = data[dedup_max_i] if (dedup_max_i is not None and dedup_max_i < len(data)) else ""
#                         existing = best.get(key)
#                         if existing is None or val > existing[0]:
#                             best[key] = (val, slim_row)
#                     else:
#                         rows.append(slim_row)
#     final_rows = [v[1] for v in best.values()] if use_dedup else rows
#     return pd.DataFrame(final_rows, columns=header) if final_rows else pd.DataFrame(columns=header or [])


# def _process_month_gap(month: pd.Timestamp, duid_to_region: dict) -> pd.DataFrame:
#     y, m, ym = month.year, month.month, month.strftime("%Y%m")
#     base      = _AEMO_MMS_BASE.format(year=y, month=m)
#     target_ym = month.strftime("%Y/%m")
#     bdo_url = f"{base}/PUBLIC_DVD_BIDDAYOFFER_{ym}010000.zip"
#     bdo_zip = RAW_DIR / f"TEMP_BIDDAYOFFER_{ym}.zip"
#     print(f"    Downloading BIDDAYOFFER_{ym} (~128 MB)...", flush=True)
#     if not _stream_download(bdo_url, bdo_zip):
#         raise ValueError(f"Could not download BIDDAYOFFER for {month:%Y-%m}")
#     bdo_keep = {"DUID", "BIDTYPE", "SETTLEMENTDATE", "OFFERDATE"} | set(PRICE_COLS)
#     day = _parse_mms_energy_zip(bdo_zip, keep_cols=bdo_keep)
#     bdo_zip.unlink(missing_ok=True)
#     day.columns = [c.upper() for c in day.columns]
#     day["TRADE_DAY"] = pd.to_datetime(day["SETTLEMENTDATE"], errors="coerce").dt.normalize()
#     if "OFFERDATE" in day.columns:
#         day["OFFERDATE"] = pd.to_datetime(day["OFFERDATE"], errors="coerce")
#         day = day.sort_values("OFFERDATE").drop_duplicates(subset=["DUID", "TRADE_DAY"], keep="last")
#     for c in PRICE_COLS:
#         if c in day.columns:
#             day[c] = pd.to_numeric(day[c], errors="coerce")
#     bpo_keep = ({"DUID", "BIDTYPE", "TRADINGDATE", "OFFERDATETIME", "PERIODID", "MAXAVAIL"} | set(BAND_COLS))
#     per_parts = []
#     if month < GAP_SPLIT_START:
#         bpo_url = f"{base}/PUBLIC_DVD_BIDPEROFFER_{ym}010000.zip"
#         bpo_zip = RAW_DIR / f"TEMP_BIDPEROFFER_{ym}.zip"
#         try:
#             mb = int(requests.head(bpo_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15).headers.get("Content-Length", 0)) // 1024 // 1024
#         except Exception:
#             mb = "?"
#         print(f"    Downloading BIDPEROFFER_{ym} (~{mb} MB)...", flush=True)
#         if _stream_download(bpo_url, bpo_zip):
#             part = _parse_mms_energy_zip(bpo_zip, keep_cols=bpo_keep, filter_ym=target_ym,
#                                          dedup_key=("DUID", "TRADINGDATE", "PERIODID"), dedup_max="OFFERDATETIME")
#             bpo_zip.unlink(missing_ok=True)
#             if not part.empty:
#                 per_parts.append(part)
#         else:
#             bpo_zip.unlink(missing_ok=True)
#     else:
#         for idx in (1, 2):
#             bpo_url = f"{base}/PUBLIC_DVD_BIDPEROFFER{idx}_{ym}010000.zip"
#             bpo_zip = RAW_DIR / f"TEMP_BIDPEROFFER{idx}_{ym}.zip"
#             try:
#                 mb = int(requests.head(bpo_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15).headers.get("Content-Length", 0)) // 1024 // 1024
#             except Exception:
#                 mb = "?"
#             print(f"    Downloading BIDPEROFFER{idx}_{ym} (~{mb} MB)...", flush=True)
#             if _stream_download(bpo_url, bpo_zip):
#                 part = _parse_mms_energy_zip(bpo_zip, keep_cols=bpo_keep, filter_ym=target_ym,
#                                              dedup_key=("DUID", "TRADINGDATE", "PERIODID"), dedup_max="OFFERDATETIME")
#                 bpo_zip.unlink(missing_ok=True)
#                 if not part.empty:
#                     per_parts.append(part)
#             else:
#                 bpo_zip.unlink(missing_ok=True)
#                 print(f"    BIDPEROFFER{idx}_{ym} not available, skipping.", flush=True)
#     if not per_parts:
#         raise ValueError(f"No BIDPEROFFER data available for {month:%Y-%m}")
#     per = pd.concat(per_parts, ignore_index=True)
#     per.columns = [c.upper() for c in per.columns]
#     per["TRADINGDATE"]   = pd.to_datetime(per["TRADINGDATE"],   errors="coerce")
#     per["OFFERDATETIME"] = pd.to_datetime(per["OFFERDATETIME"], errors="coerce")
#     per["PERIODID"]      = pd.to_numeric(per["PERIODID"],       errors="coerce")
#     per["SETTLEMENTDATE"] = (per["TRADINGDATE"].dt.normalize() + pd.to_timedelta(4 * 60 + per["PERIODID"] * 5, unit="min"))
#     per["TRADE_DAY"] = per["TRADINGDATE"].dt.normalize()
#     for c in BAND_COLS + ["MAXAVAIL"]:
#         per[c] = pd.to_numeric(per[c], errors="coerce").fillna(0.0).clip(lower=0.0)
#     price_cols_present = [c for c in PRICE_COLS if c in day.columns]
#     merged = per.merge(day[["DUID", "TRADE_DAY"] + price_cols_present], on=["DUID", "TRADE_DAY"], how="left")
#     for c in PRICE_COLS:
#         if c in merged.columns:
#             merged[c] = pd.to_numeric(merged[c], errors="coerce")
#         else:
#             merged[c] = np.nan
#     merged["REGIONID"] = merged["DUID"].map(duid_to_region)
#     merged = merged.dropna(subset=["REGIONID"])
#     for tier, lo, hi in PRICE_TIERS:
#         merged[f"mw_{tier}"] = sum(
#             merged[f"BANDAVAIL{i}"].where((merged[f"PRICEBAND{i}"] > lo) & (merged[f"PRICEBAND{i}"] <= hi), 0.0)
#             for i in range(1, 11))
#     sec = (merged["SETTLEMENTDATE"] - merged["OFFERDATETIME"]).dt.total_seconds()
#     merged["rebid_count"] = sec.between(0, 1800).astype(np.int8)
#     rename_map = {f"mw_{t}": f"bidstack_mw_{t}" for t in TIER_NAMES}
#     rename_map["MAXAVAIL"] = "bidstack_mw_total"
#     rename_map["rebid_count"] = "bidstack_rebid_count"
#     merged = merged.rename(columns=rename_map)
#     feat_cols = list(rename_map.values())
#     agg = merged.groupby(["SETTLEMENTDATE", "REGIONID"])[feat_cols].sum().reset_index()
#     slim = agg.melt(id_vars=["SETTLEMENTDATE", "REGIONID"], value_vars=feat_cols)
#     slim["col"] = slim["variable"] + "_" + slim["REGIONID"]
#     return slim[["SETTLEMENTDATE", "col", "value"]]


# def _process_month(start: str, end: str, duid_to_region: dict) -> pd.DataFrame:
#     per = nemosis.dynamic_data_compiler(
#         start_time=start, end_time=end, table_name="BIDPEROFFER_D",
#         raw_data_location=str(RAW_DIR),
#         select_columns=["SETTLEMENTDATE", "DUID", "BIDTYPE", "LASTCHANGED"] + BAND_COLS + ["MAXAVAIL"],
#         filter_cols=["BIDTYPE"], filter_values=[["ENERGY"]],
#         fformat="feather", keep_csv=False, parse_data_types=False,
#     )
#     day = nemosis.dynamic_data_compiler(
#         start_time=start, end_time=end, table_name="BIDDAYOFFER_D",
#         raw_data_location=str(RAW_DIR),
#         select_columns=["SETTLEMENTDATE", "DUID", "BIDTYPE"] + PRICE_COLS,
#         filter_cols=["BIDTYPE"], filter_values=[["ENERGY"]],
#         fformat="feather", keep_csv=False, parse_data_types=False,
#     )
#     per.columns = [c.upper() for c in per.columns]
#     day.columns = [c.upper() for c in day.columns]
#     per["SETTLEMENTDATE"] = pd.to_datetime(per["SETTLEMENTDATE"])
#     if "LASTCHANGED" not in per.columns:
#         per["LASTCHANGED"] = pd.NaT
#     per["LASTCHANGED"] = pd.to_datetime(per["LASTCHANGED"], errors="coerce")
#     day["SETTLEMENTDATE"] = pd.to_datetime(day["SETTLEMENTDATE"])
#     for c in BAND_COLS + ["MAXAVAIL"]:
#         per[c] = pd.to_numeric(per[c], errors="coerce").fillna(0.0).clip(lower=0.0)
#     for c in PRICE_COLS:
#         day[c] = pd.to_numeric(day[c], errors="coerce")
#     per["TRADE_DAY"] = _trading_day(per["SETTLEMENTDATE"])
#     day = day.rename(columns={"SETTLEMENTDATE": "TRADE_DAY"})
#     merged = per.merge(day[["DUID", "TRADE_DAY"] + PRICE_COLS], on=["DUID", "TRADE_DAY"], how="left")
#     merged["REGIONID"] = merged["DUID"].map(duid_to_region)
#     merged = merged.dropna(subset=["REGIONID"])
#     for tier, lo, hi in PRICE_TIERS:
#         merged[f"mw_{tier}"] = sum(
#             merged[f"BANDAVAIL{i}"].where((merged[f"PRICEBAND{i}"] > lo) & (merged[f"PRICEBAND{i}"] <= hi), 0.0)
#             for i in range(1, 11))
#     sec = (merged["SETTLEMENTDATE"] - merged["LASTCHANGED"]).dt.total_seconds()
#     merged["rebid_count"] = sec.between(0, 1800).astype(np.int8)
#     rename_map = {f"mw_{t}": f"bidstack_mw_{t}" for t in TIER_NAMES}
#     rename_map["MAXAVAIL"] = "bidstack_mw_total"
#     rename_map["rebid_count"] = "bidstack_rebid_count"
#     merged = merged.rename(columns=rename_map)
#     feat_cols = list(rename_map.values())
#     agg = merged.groupby(["SETTLEMENTDATE", "REGIONID"])[feat_cols].sum().reset_index()
#     slim = agg.melt(id_vars=["SETTLEMENTDATE", "REGIONID"], value_vars=feat_cols)
#     slim["col"] = slim["variable"] + "_" + slim["REGIONID"]
#     return slim[["SETTLEMENTDATE", "col", "value"]]


# def main():
#     data_start = pd.Timestamp("2018/01/01")
#     data_end   = pd.Timestamp("2026/01/01")
#     print("Fetching DUID → region lookup...", flush=True)
#     duid_to_region = _fetch_duid_region()
#     print(f"  {len(duid_to_region)} DUIDs mapped.", flush=True)
#     for f in RAW_DIR.glob("*DUDETAILSUMMARY*"):
#         if f.is_file():
#             f.unlink()
#     processed_path = Path("Processed_data/9_bidstack.csv")
#     already_processed = set()
#     if processed_path.exists():
#         existing_idx = pd.read_csv(processed_path, usecols=[0], parse_dates=True).iloc[:, 0]
#         already_processed = set(pd.to_datetime(existing_idx).dt.to_period("M"))
#         print(f"  Found {len(already_processed)} already-processed month(s) — will skip.", flush=True)
#     total_months = pd.date_range(data_start, data_end - pd.Timedelta(days=1), freq="MS")
#     for i, month in enumerate(total_months):
#         if month.to_period("M") in already_processed:
#             print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} skipping (already processed).", flush=True)
#             continue
#         is_gap = GAP_START <= month < GAP_END
#         start  = month.strftime("%Y/%m/%d %H:%M:%S")
#         end    = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M:%S")
#         label  = "(gap: direct download)" if is_gap else ""
#         print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} fetching {label}...", flush=True)
#         try:
#             if is_gap:
#                 slim = _process_month_gap(month, duid_to_region)
#             else:
#                 slim = _process_month(start, end, duid_to_region)
#             _cleanup_month_raw(month)
#             _cleanup_month_raw(month - pd.offsets.MonthBegin(1))
#             month_df = slim.pivot_table(index="SETTLEMENTDATE", columns="col", values="value", aggfunc="first")
#             month_df.columns.name = None
#             month_df = month_df[(month_df.index >= data_start) & (month_df.index < data_end)]
#             month_df.index.name = "SETTLEMENTDATE"
#             if processed_path.exists():
#                 existing_processed = pd.read_csv(processed_path, index_col=0, parse_dates=True)
#                 existing_processed.index = pd.to_datetime(existing_processed.index, format="mixed")
#                 combined = pd.concat([existing_processed, month_df])
#                 combined = combined[~combined.index.duplicated(keep="last")].sort_index()
#             else:
#                 combined = month_df
#             combined = combined[sorted(combined.columns, key=lambda c: (c.rsplit("_", 1)[-1], c.rsplit("_", 1)[0]))]
#             combined.to_csv(processed_path)
#             already_processed.add(month.to_period("M"))
#             print(f"  {month:%Y-%m} saved.", flush=True)
#         except Exception:
#             print(f"  WARNING: {month:%Y-%m} failed:\n{traceback.format_exc()}", flush=True)
#     return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail() if processed_path.exists() else None

# main()


In [52]:
def _process_month(start: str, end: str) -> pd.DataFrame:
    band_cols  = [f"BANDAVAIL{i}" for i in range(1, 11)]
    price_cols = [f"PRICEBAND{i}" for i in range(1, 11)]

    per = nemosis.dynamic_data_compiler(
        start_time=start,
        end_time=end,
        table_name="BIDPEROFFER_D",
        raw_data_location="Pre_processing/temporary_cache",
        select_columns=["INTERVAL_DATETIME", "DUID", "BIDTYPE"] + band_cols + ["MAXAVAIL"],
        filter_cols=["BIDTYPE"],
        filter_values=[["ENERGY"]],
        fformat="feather",
        keep_csv=False,
        parse_data_types=False,
    )
    day = nemosis.dynamic_data_compiler(
        start_time=start,
        end_time=end,
        table_name="BIDDAYOFFER_D",
        raw_data_location="Pre_processing/temporary_cache",
        select_columns=["SETTLEMENTDATE", "DUID", "BIDTYPE"] + price_cols,
        filter_cols=["BIDTYPE"],
        filter_values=[["ENERGY"]],
        fformat="feather",
        keep_csv=False,
        parse_data_types=False,
    )

    per.columns = [c.upper() for c in per.columns]
    day.columns = [c.upper() for c in day.columns]

    # INTERVAL_DATETIME is the actual 5-min dispatch timestamp
    per = per.rename(columns={"INTERVAL_DATETIME": "SETTLEMENTDATE"})
    per["SETTLEMENTDATE"] = pd.to_datetime(per["SETTLEMENTDATE"])
    day["SETTLEMENTDATE"] = pd.to_datetime(day["SETTLEMENTDATE"]).dt.normalize()

    for c in band_cols + ["MAXAVAIL"]:
        per[c] = pd.to_numeric(per[c], errors="coerce")
    for c in price_cols:
        day[c] = pd.to_numeric(day[c], errors="coerce")

    d = per["SETTLEMENTDATE"].dt.normalize()
    per["TRADE_DAY"] = d.where(per["SETTLEMENTDATE"].dt.hour >= 4, d - pd.Timedelta(days=1))
    day = day.rename(columns={"SETTLEMENTDATE": "TRADE_DAY"})

    merged = per.merge(day[["DUID", "TRADE_DAY"] + price_cols], on=["DUID", "TRADE_DAY"], how="left")

    return merged[["SETTLEMENTDATE", "DUID", "MAXAVAIL"] + band_cols + price_cols]


def main():
    data_start = pd.Timestamp("2018/01/01")
    data_end   = pd.Timestamp("2026/01/01")

    processed_path = Path("Processed_data/9_bidstack.csv")
    already_processed = set()
    if processed_path.exists():
        existing_dates = pd.read_csv(processed_path, usecols=["SETTLEMENTDATE"])
        already_processed = set(pd.to_datetime(existing_dates["SETTLEMENTDATE"]).dt.to_period("M"))
        print(f"Found {len(already_processed)} already-processed month(s) — will skip.", flush=True)

    total_months = pd.date_range(data_start, data_end - pd.Timedelta(days=1), freq="MS")

    for i, month in enumerate(total_months):
        if month.to_period("M") in already_processed:
            print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} skipping.", flush=True)
            continue

        start = month.strftime("%Y/%m/%d %H:%M:%S")
        end   = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1, hours=4)).strftime("%Y/%m/%d %H:%M:%S")
        print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} fetching...", flush=True)

        try:
            month_df = _process_month(start, end)
            trade_day = month_df["SETTLEMENTDATE"].dt.normalize()
            trade_day = trade_day.where(month_df["SETTLEMENTDATE"].dt.hour >= 4, trade_day - pd.Timedelta(days=1))
            month_df = month_df[(trade_day >= month) & (trade_day < month + pd.offsets.MonthBegin(1))]
            write_header = not processed_path.exists()
            month_df.to_csv(processed_path, mode="a", header=write_header, index=False)
            already_processed.add(month.to_period("M"))
            print(f"  saved ({len(month_df):,} rows).", flush=True)
        except Exception:
            print(f"  WARNING: {month:%Y-%m} failed, skipping:\n{traceback.format_exc()}", flush=True)

    return pd.read_csv(processed_path).tail() if processed_path.exists() else None

main()


Found 3 already-processed month(s) — will skip.
  1/96 2018-01 skipping.
  2/96 2018-02 skipping.
  3/96 2018-03 skipping.
  4/96 2018-04 fetching...
INFO: Compiling data for table BIDPEROFFER_D
INFO: Downloading data for table BIDPEROFFER_D, year 2018, month 03
INFO: Creating feather file for BIDPEROFFER_D, 2018, 03
INFO: Downloading data for table BIDPEROFFER_D, year 2018, month 04
INFO: Creating feather file for BIDPEROFFER_D, 2018, 04
INFO: Downloading data for table BIDPEROFFER_D, year 2018, month 05
INFO: Creating feather file for BIDPEROFFER_D, 2018, 05
INFO: Returning BIDPEROFFER_D.
INFO: Compiling data for table BIDDAYOFFER_D
INFO: Downloading data for table BIDDAYOFFER_D, year 2018, month 03
INFO: Creating feather file for BIDDAYOFFER_D, 2018, 03
INFO: Downloading data for table BIDDAYOFFER_D, year 2018, month 04
INFO: Creating feather file for BIDDAYOFFER_D, 2018, 04
INFO: Returning BIDDAYOFFER_D.
  saved (2,171,902 rows).
  5/96 2018-05 fetching...
INFO: Compiling data for 

KeyboardInterrupt: 